# Week 07 - SQLite CRUD and Validation

| 항목 | 내용 |
|---|---|
| 이름 | 이성민 |
| 학과 | 소프트웨어융합과 |
| 학년 | 2학년 |
| 학번 | 2151050 |
| 시트 기준 열 | 417_DB-CRUD-gradio 서비스(점수반영) Quiz-Test |

이 노트북은 과제 요구사항을 학습용으로 재구성한 설명형 산출물이다. 원본 코드를 그대로 복사하지 않고, 같은 개념을 작은 로컬 예제로 다시 구현한다.

## 목표

CSV, SQLite, 검증 함수, CRUD 함수를 하나의 서비스 로직으로 연결한다.

모든 코드는 외부 서비스 접속 없이 실행되도록 구성했다. 실제 API나 웹사이트를 사용할 때는 같은 처리 흐름에서 데이터 입력 부분만 교체하면 된다.

## 1. CSV에서 SQLite로 적재

4월 17일 과제의 핵심은 CSV 데이터, SQLite 저장소, CRUD 함수, 입력 검증을 연결하는 것이다. 여기서는 메모리 SQLite를 사용해 파일 없이 실행되도록 만든다.

In [1]:
import sqlite3
import pandas as pd

initial_customers = pd.DataFrame([
    {"customer_id": "C001", "gender": "F", "payment": "card", "region": "Seoul", "grade": "gold", "satisfaction": 5, "age": 22},
    {"customer_id": "C002", "gender": "M", "payment": "cash", "region": "Busan", "grade": "silver", "satisfaction": 3, "age": 27},
    {"customer_id": "C003", "gender": "F", "payment": "card", "region": "Incheon", "grade": "bronze", "satisfaction": 4, "age": 24},
])

conn = sqlite3.connect(":memory:")
initial_customers.to_sql("customers", conn, index=False, if_exists="replace")

pd.read_sql_query("SELECT * FROM customers", conn)

,customer_id,gender,payment,region,grade,satisfaction,age
0,C001,F,card,Seoul,gold,5,22
1,C002,M,cash,Busan,silver,3,27
2,C003,F,card,Incheon,bronze,4,24


## 2. 입력 검증

데이터베이스에 넣기 전에 값의 범위를 확인해야 한다. 만족도는 1부터 5까지, 나이는 현실적인 범위로 제한한다.

In [2]:
def validate_customer(row: dict) -> dict:
    required = {"customer_id", "gender", "payment", "region", "grade", "satisfaction", "age"}
    missing = required - set(row)
    if missing:
        raise ValueError(f"missing fields: {sorted(missing)}")
    if row["gender"] not in {"F", "M"}:
        raise ValueError("gender must be F or M")
    if not 1 <= int(row["satisfaction"]) <= 5:
        raise ValueError("satisfaction must be between 1 and 5")
    if not 10 <= int(row["age"]) <= 100:
        raise ValueError("age must be between 10 and 100")
    return row

validate_customer({"customer_id": "C004", "gender": "M", "payment": "app", "region": "Daegu", "grade": "gold", "satisfaction": 4, "age": 21})

{'customer_id': 'C004',
 'gender': 'M',
 'payment': 'app',
 'region': 'Daegu',
 'grade': 'gold',
 'satisfaction': 4,
 'age': 21}

## 3. CRUD 함수

CRUD는 Create, Read, Update, Delete다. UI가 붙더라도 내부에서는 아래 함수들이 호출된다.

In [3]:
def create_customer(conn, row: dict) -> str:
    row = validate_customer(row)
    conn.execute(
        "INSERT INTO customers VALUES (?, ?, ?, ?, ?, ?, ?)",
        (row["customer_id"], row["gender"], row["payment"], row["region"], row["grade"], row["satisfaction"], row["age"]),
    )
    conn.commit()
    return row["customer_id"]

def read_customers(conn, region: str | None = None) -> pd.DataFrame:
    if region:
        return pd.read_sql_query("SELECT * FROM customers WHERE region = ?", conn, params=(region,))
    return pd.read_sql_query("SELECT * FROM customers", conn)

def update_satisfaction(conn, customer_id: str, satisfaction: int) -> int:
    validate_customer({
        "customer_id": customer_id,
        "gender": "F",
        "payment": "card",
        "region": "Seoul",
        "grade": "gold",
        "satisfaction": satisfaction,
        "age": 20,
    })
    cur = conn.execute("UPDATE customers SET satisfaction = ? WHERE customer_id = ?", (satisfaction, customer_id))
    conn.commit()
    return cur.rowcount

def delete_customer(conn, customer_id: str) -> int:
    cur = conn.execute("DELETE FROM customers WHERE customer_id = ?", (customer_id,))
    conn.commit()
    return cur.rowcount

create_customer(conn, {"customer_id": "C004", "gender": "M", "payment": "app", "region": "Daegu", "grade": "gold", "satisfaction": 4, "age": 21})
update_count = update_satisfaction(conn, "C002", 5)
current = read_customers(conn)

assert update_count == 1
assert "C004" in set(current["customer_id"])
current

,customer_id,gender,payment,region,grade,satisfaction,age
0,C001,F,card,Seoul,gold,5,22
1,C002,M,cash,Busan,silver,5,27
2,C003,F,card,Incheon,bronze,4,24
3,C004,M,app,Daegu,gold,4,21


## 4. 서비스용 요약 함수

Gradio 같은 UI는 표와 문자열을 반환하는 함수를 화면 컴포넌트에 연결한다. 아래 함수는 현재 DB 상태를 요약해 UI 출력값처럼 사용할 수 있다.

In [4]:
def service_summary(conn) -> str:
    df = read_customers(conn)
    avg = df["satisfaction"].mean()
    top_region = df["region"].value_counts().idxmax()
    return f"customers={len(df)}, avg_satisfaction={avg:.2f}, top_region={top_region}"

summary_text = service_summary(conn)
removed = delete_customer(conn, "C003")

assert removed == 1
assert "customers=" in summary_text
print(summary_text)
display(read_customers(conn))

customers=4, avg_satisfaction=4.50, top_region=Seoul


,customer_id,gender,payment,region,grade,satisfaction,age
0,C001,F,card,Seoul,gold,5,22
1,C002,M,cash,Busan,silver,5,27
2,C004,M,app,Daegu,gold,4,21
